## **Title: Swiggy Food Delivery Dataset Using Python**

#### **Objectives & Aim** 

To analyze Swiggy's restaurant and food delivery dataset to uncover market insights, customer preferences, pricing trends, and business opportunities for stakeholders including:

* Restaurant owners/partners
* Platform strategists
* Marketing teams
* Food entrepreneurs

### **Import Libraries** 

In [1]:
import pandas as pd              # Used for data handling and analysis (tables, CSV, Excel)
import numpy as np               # Used for numerical operations and arrays

### **Load Dataset**

In [2]:
df = pd.read_csv(r"Swiggy Raw Dataset.csv")
df.head()

,State,City,Order Date,Restaurant Name,Location,Category,Dish Name,Price (INR),Rating,Rating Count
0,Karnataka,Bengaluru,6/29/2025,Anand Sweets & Savouries,Rajarajeshwari Nagar,Snack,Butter Murukku-200gm,133.9,4.0,0
1,Karnataka,Bengaluru,4/3/2025,Srinidhi Sagar Deluxe,Kengeri,Recommended,Badam Milk,52.0,4.5,25
2,Karnataka,Bengaluru,1/15/2025,Srinidhi Sagar Deluxe,Kengeri,Recommended,Chow Chow Bath,117.0,4.7,48
3,Karnataka,Bengaluru,4/17/2025,Srinidhi Sagar Deluxe,Kengeri,Recommended,Kesari Bath,65.0,4.6,65
4,Karnataka,Bengaluru,3/13/2025,Srinidhi Sagar Deluxe,Kengeri,Recommended,Mix Raitha,130.0,4.0,0


### **Basic Dataset Information**

In [3]:
df.shape

(197430, 10)

In [4]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197430 entries, 0 to 197429
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   State            197430 non-null  object 
 1   City             197430 non-null  object 
 2   Order Date       197430 non-null  object 
 3   Restaurant Name  197430 non-null  object 
 4   Location         197430 non-null  object 
 5   Category         197430 non-null  object 
 6   Dish Name        197430 non-null  object 
 7   Price (INR)      197430 non-null  float64
 8   Rating           197430 non-null  float64
 9   Rating Count     197430 non-null  int64  
dtypes: float64(2), int64(1), object(7)
memory usage: 15.1+ MB


,Price (INR),Rating,Rating Count
count,197430.000000,197430.000000,197430.000000
mean,268.512920,4.341582,28.321805
std,219.338363,0.422585,87.542593
min,0.950000,1.500000,0.000000
25%,139.000000,4.300000,0.000000
50%,229.000000,4.400000,2.000000
75%,329.000000,4.500000,15.000000
max,8000.000000,5.000000,999.000000


**Clean column names (remove extra spaces, convert to lowercase)**

In [5]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print("After cleaning column names:")
print(df.columns.tolist())

After cleaning column names:
['state', 'city', 'order_date', 'restaurant_name', 'location', 'category', 'dish_name', 'price_(inr)', 'rating', 'rating_count']


**Check data types and missing values**


In [6]:
print("Data Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Data Types:
state               object
city                object
order_date          object
restaurant_name     object
location            object
category            object
dish_name           object
price_(inr)        float64
rating             float64
rating_count         int64
dtype: object

Missing Values:
state              0
city               0
order_date         0
restaurant_name    0
location           0
category           0
dish_name          0
price_(inr)        0
rating             0
rating_count       0
dtype: int64


**Standardize date format**

In [7]:
# This code converts the order_date column into a proper datetime format and safely handles invalid dates.
df['order_date'] = pd.to_datetime(df['order_date'], format='%m/%d/%Y', errors='coerce')

**Extract date features for analysis**

In [8]:
# This code extracts the year, month, day, and weekday from the date column and creates new columns, which is useful for Exploratory Data Analysis (EDA).
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_day'] = df['order_date'].dt.day
df['order_weekday'] = df['order_date'].dt.weekday  # 0=Monday, 6=Sunday

**Handle missing values**

In [9]:
# For numerical columns, fill with appropriate values
df['rating'] = df['rating'].fillna(0)
df['rating_count'] = df['rating_count'].fillna(0)

# For categorical columns, fill with 'Unknown'
categorical_cols = ['state', 'city', 'restaurant_name', 'location', 'category', 'dish_name']
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

**Standardize text in categorical columns (remove extra spaces, proper case)**

In [10]:
# This code cleans and standardizes text data by removing extra spaces and converting values to title case.
def clean_text(text):
    if isinstance(text, str):
        # Remove multiple spaces, leading/trailing spaces
        text = ' '.join(text.split())
        # Convert to title case for consistency
        return text.title()
    return text

for col in categorical_cols:
    df[col] = df[col].apply(clean_text)

**Fix common spelling errors in dish names**

In [11]:
spelling_corrections = {
    'Panneer': 'Paneer',
    'Panner': 'Paneer',
    'Masla': 'Masala',
    'Chciken': 'Chicken',
    'Biriyani': 'Biryani',
    'Briyani': 'Biryani',
    'Noddles': 'Noodles',
    'Chiken': 'Chicken',
    'Roti': 'Roti',
    'Parrota': 'Paratha',
    'Parota': 'Paratha',
    'Kulcha': 'Kulcha',
    'Naan': 'Naan',
    'Raitha': 'Raita',
    'Murukku': 'Murukku',
    'Schezwan': 'Szechuan',
    'Manchurian': 'Manchurian',
    'Tandoori': 'Tandoori'
}

def correct_spelling(text):
    if isinstance(text, str):
        for wrong, correct in spelling_corrections.items():
            text = text.replace(wrong, correct)
    return text

df['dish_name'] = df['dish_name'].apply(correct_spelling)

**Clean price column**

In [12]:
# Remove any non-numeric characters and convert to float
df['price_(inr)'] = df['price_(inr)'].astype(str).str.replace('[^\d.]', '', regex=True)
df['price_(inr)'] = pd.to_numeric(df['price_(inr)'], errors='coerce')

# Fill missing prices with median of the category
df['price_(inr)'] = df.groupby('category')['price_(inr)'].transform(
    lambda x: x.fillna(x.median())
)

**Create new features**

In [13]:
# Price categories
df['price_category'] = pd.cut(
    df['price_(inr)'],
    bins=[0, 100, 200, 300, 500, float('inf')],
    labels=['Budget', 'Affordable', 'Moderate', 'Premium', 'Luxury']
)

# Rating categories
df['rating_category'] = pd.cut(
    df['rating'],
    bins=[0, 2, 3, 4, 5],
    labels=['Poor', 'Average', 'Good', 'Excellent'],
    include_lowest=True
)

**Standardize category names**

In [14]:
category_mapping = {
    'Recommended': 'Recommended',
    'North Indian Gravy': 'North Indian',
    'North Indian Rice': 'North Indian',
    'Manchurian & Chilly': 'Indo-Chinese',
    'South Indian Dishes': 'South Indian',
    'Chinese Rice': 'Chinese',
    'Tandoori Breads': 'Breads',
    'Special Ice Cream': 'Desserts',
    'Fresh Fruit Juice': 'Beverages',
    'Flavoured Milkshakes': 'Beverages',
    'Noodles': 'Chinese',
    'Kadai': 'North Indian',
    'North Special Dishes': 'North Indian',
    'Meals': 'Meals',
    'Soups': 'Starters',
    'Ice Creams In Cup': 'Desserts',
    'Fresh Fruit Milkshakes': 'Beverages',
    'Raitha': 'Sides',
    'Salad & Papad': 'Sides',
    'Beverages': 'Beverages',
    'Sweets': 'Desserts',
    'Ice Cream Soda': 'Beverages',
    'Dosas': 'South Indian',
    'Chaats': 'Street Food',
    'Biryani': 'Rice Dishes',
    'Starter': 'Starters',
    'Main Course': 'Main Course',
    'Egg': 'Egg Dishes',
    'Bread': 'Breads',
    'Rice': 'Rice Dishes',
    'Item': 'General',
    'Non Veg Starter': 'Starters',
    'Tandoori Starter': 'Starters',
    'Veg Gravy': 'Vegetarian',
    'Indian Breads': 'Breads',
    'Paneer': 'Vegetarian',
    'Mushroom': 'Vegetarian',
    'Extra': 'Sides',
    'Swadh Signature': 'Special',
    'Breakfast Combos': 'Breakfast',
    'Snacks': 'Snacks',
    'Snack': 'Snacks',
    'Pot Rice': 'Rice Dishes',
    'Dessert': 'Desserts',
    'Drinks': 'Beverages',
    'Extra Dip & Crispy Noodles': 'Sides',
    'Veg Pizza': 'Pizza',
    'Non Veg Pizza': 'Pizza',
    'Thin n Crispy Pizzas': 'Pizza',
    'Hut Café': 'Beverages',
    'Pasta': 'Pasta',
    'One Plus One Medium @649': 'Combos',
    'Garlic Bread': 'Sides',
    'Appetizer': 'Starters',
    'Oh-so-Cheesy (New Launch)': 'Pizza',
    'Stuffed Garlic Breads': 'Sides',
    'Pepperoni Pizza - Chef\'s Special': 'Pizza',
    'Loaded Nachos': 'Starters',
    'Cheesy Pizza Mania Box': 'Pizza',
    'Guiltfree By Eatfit!': 'Healthy',
    'Cheese Burst Pizza': 'Pizza',
    'Kids Special - Pizza Party': 'Kids',
    'House Special Garlic Breads!': 'Sides',
    'Signature Garlic Knots': 'Sides',
    'Pastas': 'Pasta',
    'Beverages': 'Beverages',
    'Krispy Kreme Doughnuts and Coffee': 'Desserts',
    'Pasta Lunch Box': 'Pasta',
    'Desserts': 'Desserts',
    'Half & Half Pizza': 'Pizza',
    'Crazy Deals': 'Combos',
    'Royal 4in1 Pizza': 'Pizza',
    'Dips': 'Sides',
    'ROLLS': 'Rolls',
    'BURGERS': 'Burgers',
    'ALL DAY LUNCH SPECIAL MEAL BOX (SAVE RS 115)': 'Combos',
    'PERI PERI CHICKEN STRIPS & LEG PC (UP TO 25% OFF)': 'Chicken',
    'EPIC VALUE MEALS FOR 1-2 (UP TO 22% OFF)': 'Combos',
    'EPIC SAVINGS BUCKET FOR 3-4 (UP TO 32% OFF)': 'Combos',
    'RICE BOWLZ': 'Rice Dishes',
    'HOT & CRISPY CHICKEN & WINGS': 'Chicken',
    'BONELESS CHICKEN POPCORN': 'Chicken',
    'GRILLED SMOKY RED CHICKEN': 'Chicken',
    'FLAT 179 SPECIAL MEALS/BUCKET': 'Combos',
    'SIDES AND DIPS': 'Sides',
    'DESSERTS & BEVERAGES': 'Desserts',
    'LATE NIGHT SPECIALS (STARTING AT 199)': 'Combos',
    'Ganesha Festival Special': 'Festival Special',
    'Gift Hampers': 'Gifts',
    'Luxe Candles': 'Gifts',
    'Assorted Sweets': 'Sweets',
    'Sweets': 'Sweets',
    'Bengali Sweets': 'Sweets',
    'Namkeen': 'Snacks',
    'Muruku': 'Snacks',
    'Powder': 'Groceries',
    'Sticks': 'Snacks',
    'Cookies And Biscuits': 'Bakery',
    'Packed': 'Packaged',
    'Breads': 'Bakery',
    'Cakes': 'Bakery',
    'Rusk And Khari': 'Bakery',
    'Chocolates': 'Sweets',
    'Doughnut': 'Bakery',
    'Crossiants  & Rolls': 'Bakery',
    'Saffron': 'Groceries',
    'Death by Chocolate Sundaes': 'Desserts',
    'Chocolate Sundaes': 'Desserts',
    'Polar Bear Originals': 'Desserts',
    'Fruity Delight': 'Desserts',
    'Ice Cream Cakes': 'Desserts',
    'Faloodas': 'Desserts',
    'Fundaes': 'Desserts',
    'Dairy-Free Sorbets': 'Desserts',
    'Naturally Good Sundaes': 'Desserts',
    'Asian Fusion  Sundaes': 'Desserts',
    'Biscoff Cookie Sundaes': 'Desserts',
    'Family  Sundaes': 'Desserts',
    'Coffee Sundaes': 'Desserts',
    'Sandwiches': 'Snacks',
    'Craft Hot Chocolate': 'Beverages',
    'Ice Cream Shake and Sandwich Combo': 'Combos',
    'Cookies': 'Bakery',
    'Extras': 'Sides',
    'Monsoon Combos': 'Combos',
    'Double Scoops, Double Smiles': 'Desserts',
    'Ganesh Chaturthi Festive Combos': 'Festival Special',
    'Newly Launched': 'New',
    '100 ml Ice creams': 'Desserts',
    '300 Ml Ice Creams': 'Desserts',
    '500 ml Ice creams': 'Desserts',
    '750 ml Ice creams': 'Desserts',
    'No Added Sugar range': 'Healthy'
}

df['standardized_category'] = df['category'].map(category_mapping)
df['standardized_category'] = df['standardized_category'].fillna('Other')

**Check and clean location names**

In [15]:
df['location'] = df['location'].str.title()
df['location'] = df['location'].str.replace('Upnagar', 'Upanagar', regex=False)
df['location'] = df['location'].str.replace('Satellite Town', 'Satellite', regex=False)

**Remove duplicates (considering all columns)**

In [16]:
initial_rows = len(df)
df = df.drop_duplicates()
removed_duplicates = initial_rows - len(df)
print(f"Removed {removed_duplicates} duplicate rows")

Removed 29 duplicate rows


**Generate cleaning report**

In [17]:
print(f"Original dataset shape: {initial_rows} rows")
print(f"Cleaned dataset shape: {len(df)} rows")
print(f"Duplicates removed: {removed_duplicates}")
print("\nData Types After Cleaning:")
print(df.dtypes)

Original dataset shape: 197430 rows
Cleaned dataset shape: 197401 rows
Duplicates removed: 29

Data Types After Cleaning:
state                            object
city                             object
order_date               datetime64[ns]
restaurant_name                  object
location                         object
category                         object
dish_name                        object
price_(inr)                     float64
rating                          float64
rating_count                      int64
order_year                        int32
order_month                       int32
order_day                         int32
order_weekday                     int32
price_category                 category
rating_category                category
standardized_category            object
dtype: object


In [18]:
print("\nMissing Values After Cleaning:")
print(df.isnull().sum())


Missing Values After Cleaning:
state                    0
city                     0
order_date               0
restaurant_name          0
location                 0
category                 0
dish_name                0
price_(inr)              0
rating                   0
rating_count             0
order_year               0
order_month              0
order_day                0
order_weekday            0
price_category           0
rating_category          0
standardized_category    0
dtype: int64


In [22]:
print("\nPrice Statistics:")
print(f"Min Price: ₹{df['price_(inr)'].min():.2f}")
print(f"Max Price: ₹{df['price_(inr)'].max():.2f}")
print(f"Average Price: ₹{df['price_(inr)'].mean():.2f}")
print(f"Median Price: ₹{df['price_(inr)'].median():.2f}")


Price Statistics:
Min Price: ₹0.95
Max Price: ₹8000.00
Average Price: ₹268.50
Median Price: ₹229.00


In [21]:
print("\nRating Statistics:")
print(f"Average Rating: {df['rating'].mean():.2f}")
print(f"Total Ratings: {df['rating_count'].sum()}")
print(f"Unique Restaurants: {df['restaurant_name'].nunique()}")
print(f"Unique Dishes: {df['dish_name'].nunique()}")
print(f"Date Range: {df['order_date'].min()} to {df['order_date'].max()}")


Rating Statistics:
Average Rating: 4.34
Total Ratings: 5591047
Unique Restaurants: 984
Unique Dishes: 53595
Date Range: 2025-01-01 00:00:00 to 2025-08-31 00:00:00


**Show sample of cleaned data**

In [20]:
# Display first 5 rows of selected columns in proper table format
df[['restaurant_name', 'dish_name', 'price_(inr)', 'rating', 'standardized_category']].head()

,restaurant_name,dish_name,price_(inr),rating,standardized_category
0,Anand Sweets & Savouries,Butter Murukku-200Gm,133.9,4.0,Snacks
1,Srinidhi Sagar Deluxe,Badam Milk,52.0,4.5,Recommended
2,Srinidhi Sagar Deluxe,Chow Chow Bath,117.0,4.7,Recommended
3,Srinidhi Sagar Deluxe,Kesari Bath,65.0,4.6,Recommended
4,Srinidhi Sagar Deluxe,Mix Raita,130.0,4.0,Recommended


### **Save the cleaned dataset**

In [ ]:
cleaned_filename = 'Copy_Swiggy_Cleaned_Dataset.csv'
df.to_csv(cleaned_filename, index=False)